# 05 · Counterfactual perturbations

Apply the seven perturbation operators to one participant's recent state and compare the resulting latent trajectories.


In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / 'src'))
import numpy as np; import pandas as pd; import matplotlib.pyplot as plt


In [ ]:
from data.synthetic_generator import generate_synthetic_cohort
from features import engineer_all_features
from states.latent_state_encoder import encode_latent_states_classical, LATENT_DIM_NAMES
from dynamics.transition_model import LatentDynamicsModel
from counterfactuals.perturbation_engine import (
    PerturbationSpec, simulate_perturbation, available_perturbations,
)

df = engineer_all_features(generate_synthetic_cohort(n_participants=30, n_days=90, seed=17))


In [ ]:
def pick(df, cols):
    present = [c for c in cols if c in df.columns]
    return df[present].to_numpy(dtype=float, na_value=0.0) if present else np.zeros((len(df), len(cols)))

W = pick(df, ['sleep_duration_hours','hrv_rmssd','resting_hr','daily_steps','recovery_score'])
B = pick(df, ['screen_time_minutes','mobility_radius_km','location_entropy','phone_unlock_count'])
C = pick(df, ['temperature_c','nighttime_temperature_c','aqi','heat_wave_flag'])
M = pick(df, ['missing_wearable_flag','missing_phone_flag','missing_survey_flag'])
P = pick(df, ['baseline_hrv','baseline_resting_hr','baseline_climate_vulnerability','baseline_resilience'])
Z = encode_latent_states_classical(W, B, C, M, P).latent


In [ ]:
pid = df['participant_id'].iloc[0]
mask = (df['participant_id'].values == pid)
z0 = Z[mask][-1]
env_cols = ['temperature_c','nighttime_temperature_c','aqi','heat_wave_flag']
beh_cols = ['screen_time_minutes','mobility_radius_km','location_entropy','phone_unlock_count']
env_last = df[mask][env_cols].tail(1).to_numpy(dtype=float, na_value=0.0)
beh_last = df[mask][beh_cols].tail(1).to_numpy(dtype=float, na_value=0.0)
env_forecast = np.tile(env_last, (14, 1))
beh_forecast = np.tile(beh_last, (14, 1))
dyn = LatentDynamicsModel(rng_seed=17)


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 6), sharey=True)
for ax, ptype in zip(axes.flat, available_perturbations()):
    spec = PerturbationSpec(perturbation_type=ptype, horizon_days=14)
    res = simulate_perturbation(z0, env_forecast, beh_forecast, spec, dynamics_model=dyn)
    diff = np.array(res.counterfactual_trajectory) - np.array(res.baseline_trajectory)
    for i, name in enumerate(LATENT_DIM_NAMES):
        ax.plot(diff[:, i], label=name)
    ax.set_title(ptype, fontsize=8); ax.axhline(0, color='grey', lw=0.5)
for ax in list(axes.flat)[len(available_perturbations()):]: ax.axis('off')
plt.tight_layout(); plt.show()
